In [1]:
# Importations

import time
from enderscope import SerialUtils, Stage 
import serial
from math import *
import threading

In [2]:
# Variables

  # Modifiable Variables

rectangle = [40,30] # in mm, x and y
coordinate_rec_one = [10,10] # coordinate left bottom of the rectangle
printer_speed = 3 # in mm/s
total_thickness = 2
location_purge_A = [150, 150,8]


  # Fixed
width_extrusion = 5 # in mm 
number_passages = min(ceil(rectangle[0] /(2 * width_extrusion)),ceil(rectangle[1] /(2 * width_extrusion)))  # computes the smallest integer that is greater than or equal to x.
thickness_1_layer =  2
number_layer = ceil(total_thickness/thickness_1_layer)
coordinate_rec_2 = [coordinate_rec_one[0]+rectangle[0], coordinate_rec_one[1]] # Coordinates for the 2nd rectangle
coordinate_rec_3 = [coordinate_rec_one[0]+ 2 *rectangle[0], coordinate_rec_one[1]]

gap = [1,1] # For the syringe pump A, a gap of (x=1, y=1)/origin

height_printert = 10 # in mm

In [3]:
# Ports

ports = SerialUtils.serial_ports() # List of ports
print (ports) 

syringe_pump_port = ports[0] # To modify
printer_port = ports[1] # To modify

s = Stage(printer_port, 115200) 
syringe_pump = serial.Serial(port= syringe_pump_port, baudrate=115200, timeout=0.01, writeTimeout=1) # Connexion pousse seringue

['COM4', 'COM5']


In [4]:
# Syringe pump : message to start
def message_start(letter_syringe_pump): 
    return(f"{letter_syringe_pump}\n".encode('utf8')) 

In [5]:
# Syringe pump : message to back off
def message_back_off():
    return b"R\n"

In [6]:
# Syringe pump : message to stop
def message_stop():
        return b"S\n"

In [7]:
# Purge 

def purge_syringe_pump(letter_syringe_pump):
    syringe_pump.write(message_start(letter_syringe_pump))
    time.sleep(2)
    syringe_pump.write(message_back_off())
    time.sleep(2)
    syringe_pump.write(message_stop())
    time.sleep(1)

purge_A = threading.Thread( target = purge_syringe_pump, args = ('A'))

def purge():
    s.move_absolute(location_purge_A[0],location_purge_A[1],location_purge_A[2])
    s.write_code(f"M400")
    purge_A.start()
    purge_A.join()

In [8]:
# Si liquide coule malgré la fin de pression : besoin de réaspirer
def liquide (): 
    syringe_pump.write(message_back_off())
    time.sleep(1) # vérifier durée pour aspiration 
    syringe_pump.write(message_stop())


In [9]:
# Give the position considering the gap

def position(letter_syringe_pump, coordinate):
    position_tube = []
    if letter_syringe_pump == "A":
        position_tube.append(coordinate[0] - gap[0])
        position_tube.append(coordinate[1] - gap[1])
        return position_tube
    
    if letter_syringe_pump == "B":
        position_tube.append(coordinate[0] + gap[0])
        position_tube.append(coordinate[1] - gap[1])
        return position_tube

    if letter_syringe_pump == "C":
        position_tube.append(coordinate[0] + gap[0])
        position_tube.append(coordinate[1] + gap[1])
        return position_tube

In [10]:
# 1 rectangle

def function_rectangle(): 

    x_rectangle = rectangle[0]
    y_rectangle = rectangle[1]

    for i in range (number_passages):
        for j in range(number_layer):
            s.write_code(f"M203 X{printer_speed}")
            s.write_code(f"M203 Y{printer_speed}")
            s.move_axis('x', x_rectangle)
            s.move_axis ('y', y_rectangle)
            s.move_axis('x', -x_rectangle)
            s.move_axis('y', - (y_rectangle -width_extrusion))

    

            s.move_axis('x', width_extrusion)
        
            x_rectangle = x_rectangle - (2 * width_extrusion)
            y_rectangle = y_rectangle - (2*width_extrusion)

    s.write_code(f"M400")

In [11]:
# Go to the position

def go (letter_syringe_pump, coordonnes_debut):
    s.move_absolute(position(letter_syringe_pump, coordonnes_debut)[0],position(letter_syringe_pump, coordonnes_debut)[1], height_printert)    
    s.write_code(f"M400") 

In [13]:
# Focus

#s.home()

# Purge

#purge()


# Rectangle with syringe pump A
#go('A', coordinate_rec_one)  
syringe_pump.write(message_start('A'))
function_rectangle() 
syringe_pump.write(message_stop())
'''
# Rectangle with syringe pump B
go('A', coordinate_rec_2)
syringe_pump.write(message_start('A'))
function_rectangle() 
syringe_pump.write(message_stop())

# Rectangle with syringe pump C
go('A', coordinate_rec_3)
syringe_pump.write(message_start('A'))
function_rectangle() 
syringe_pump.write(message_stop())
'''


SerialException: WriteFile failed (PermissionError(13, 'Accès refusé.', None, 5))